# Video Telemetry QoS Pipeline - Alert Queries

This notebook contains SQL queries designed to be used as **scheduled alerts** to monitor pipeline health and data quality.

## Alert Categories

1. **Data Freshness** - Detects stale data or pipeline failures
2. **Data Quality** - Monitors quarantine rates and data validation failures
3. **Performance Degradation** - Tracks buffering metrics and QoS trends
4. **Data Volume Anomalies** - Detects unexpected drops in data ingestion

## How to Use

1. Run each query cell to verify it works
2. Create a **SQL Alert** from each query
3. Set appropriate **refresh schedules** (recommended: every 15-30 minutes)
4. Configure **notification destinations** (email, Slack, PagerDuty)
5. Set **alert thresholds** based on your SLAs

## Alert Setup Instructions

For each query below:
- Click the three dots menu → **Create Alert**
- Set refresh schedule
- Define alert condition (e.g., "Value" > threshold)
- Add notification recipients
- Save and enable the alert

In [0]:
%sql
-- Alert: Data Freshness - Bronze Layer
-- Purpose: Detect if no new data has arrived in the last 2 hours
-- Trigger: When hours_since_last_event > 2
-- Severity: CRITICAL

SELECT 
  'BRONZE_DATA_FRESHNESS' as alert_name,
  FROM_UNIXTIME(MAX(timestampInfo.server) / 1000) as last_event_timestamp,
  ROUND((UNIX_TIMESTAMP(CURRENT_TIMESTAMP()) - MAX(timestampInfo.server) / 1000) / 3600.0, 2) as hours_since_last_event,
  COUNT(*) as total_records,
  CASE 
    WHEN ROUND((UNIX_TIMESTAMP(CURRENT_TIMESTAMP()) - MAX(timestampInfo.server) / 1000) / 3600.0, 2) > 2 THEN 'ALERT'
    ELSE 'OK'
  END as status
FROM workspace.hive_video_analytics.bronze_telemetry_raw
WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS;

In [0]:
%sql
-- Alert: High Quarantine Rate
-- Purpose: Detect when data quality issues exceed acceptable thresholds
-- Trigger: When quarantine_rate_pct > 5%
-- Severity: HIGH

WITH recent_data AS (
  SELECT COUNT(*) as bronze_count
  FROM workspace.hive_video_analytics.bronze_telemetry_raw
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 1 DAYS
),
recent_quarantine AS (
  SELECT COUNT(*) as quarantine_count
  FROM workspace.hive_video_analytics.silver_telemetry_quarantine
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 1 DAYS
)
SELECT 
  'HIGH_QUARANTINE_RATE' as alert_name,
  bronze_count,
  quarantine_count,
  ROUND((quarantine_count * 100.0 / NULLIF(bronze_count, 0)), 2) as quarantine_rate_pct,
  CASE 
    WHEN ROUND((quarantine_count * 100.0 / NULLIF(bronze_count, 0)), 2) > 5 THEN 'ALERT'
    WHEN ROUND((quarantine_count * 100.0 / NULLIF(bronze_count, 0)), 2) > 3 THEN 'WARNING'
    ELSE 'OK'
  END as status
FROM recent_data, recent_quarantine;

In [0]:
%sql
-- Alert: Data Volume Anomaly
-- Purpose: Detect significant drops in data ingestion compared to baseline
-- Trigger: When today's volume is < 50% of 7-day average
-- Severity: HIGH

WITH baseline AS (
  SELECT AVG(daily_count) as avg_daily_count
  FROM (
    SELECT eventDate, COUNT(*) as daily_count
    FROM workspace.hive_video_analytics.bronze_telemetry_raw
    WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
      AND eventDate < CURRENT_DATE()
    GROUP BY eventDate
  )
),
today_data AS (
  SELECT COUNT(*) as today_count
  FROM workspace.hive_video_analytics.bronze_telemetry_raw
  WHERE eventDate = CURRENT_DATE()
)
SELECT 
  'DATA_VOLUME_ANOMALY' as alert_name,
  ROUND(avg_daily_count, 0) as avg_daily_baseline,
  today_count as today_volume,
  ROUND((today_count * 100.0 / NULLIF(avg_daily_count, 0)), 2) as pct_of_baseline,
  CASE 
    WHEN ROUND((today_count * 100.0 / NULLIF(avg_daily_count, 0)), 2) < 50 THEN 'ALERT'
    WHEN ROUND((today_count * 100.0 / NULLIF(avg_daily_count, 0)), 2) < 70 THEN 'WARNING'
    ELSE 'OK'
  END as status
FROM baseline, today_data;

In [0]:
%sql
-- Alert: Buffering Performance Degradation
-- Purpose: Detect degradation in streaming quality (increased buffering)
-- Trigger: When avg_buffering_time_sec > 10 seconds or buffering_sessions > 15%
-- Severity: MEDIUM

WITH today_metrics AS (
  SELECT 
    AVG(buffering_time_sec) as avg_buffering_time_sec,
    SUM(CASE WHEN buffering_count > 0 THEN 1 ELSE 0 END) as sessions_with_buffering,
    COUNT(DISTINCT CONCAT(customerId, clientId, timestamp_server)) as total_sessions,
    ROUND(SUM(CASE WHEN buffering_count > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT CONCAT(customerId, clientId, timestamp_server)), 2) as buffering_sessions_pct
  FROM workspace.hive_video_analytics.silver_telemetry_enriched
  WHERE eventDate = CURRENT_DATE()
)
SELECT 
  'BUFFERING_DEGRADATION' as alert_name,
  ROUND(avg_buffering_time_sec, 2) as avg_buffering_time_sec,
  total_sessions,
  sessions_with_buffering,
  buffering_sessions_pct,
  CASE 
    WHEN avg_buffering_time_sec > 10 OR buffering_sessions_pct > 15 THEN 'ALERT'
    WHEN avg_buffering_time_sec > 7 OR buffering_sessions_pct > 10 THEN 'WARNING'
    ELSE 'OK'
  END as status
FROM today_metrics;

In [0]:
%sql
-- Alert: Gold Layer Processing Lag
-- Purpose: Detect if gold layer is falling behind silver layer
-- Trigger: When lag_hours > 1 hour
-- Severity: MEDIUM
-- Note: Gold layer is a MATERIALIZED VIEW, not a streaming table

WITH silver_latest AS (
  SELECT MAX(timestamp_server) as latest_timestamp
  FROM workspace.hive_video_analytics.silver_telemetry_enriched
),
gold_latest AS (
  SELECT MAX(eventDate) as latest_date
  FROM workspace.hive_video_analytics.gold_viewer_qos_metrics
)
SELECT 
  'GOLD_LAYER_LAG' as alert_name,
  FROM_UNIXTIME(silver_latest.latest_timestamp / 1000) as silver_latest_event,
  gold_latest.latest_date as gold_latest_date,
  DATEDIFF(CURRENT_DATE(), gold_latest.latest_date) as lag_days,
  CASE 
    WHEN DATEDIFF(CURRENT_DATE(), gold_latest.latest_date) > 1 THEN 'ALERT'
    WHEN DATEDIFF(CURRENT_DATE(), gold_latest.latest_date) > 0 THEN 'WARNING'
    ELSE 'OK'
  END as status
FROM silver_latest, gold_latest;

In [0]:
%sql
-- Alert: Quarantine Reason Spike
-- Purpose: Detect sudden increases in specific quarantine reasons
-- Trigger: When any single quarantine reason > 1000 records today
-- Severity: MEDIUM

SELECT 
  'QUARANTINE_REASON_SPIKE' as alert_name,
  quarantine_reason,
  COUNT(*) as issue_count,
  COUNT(DISTINCT customerId) as affected_customers,
  CASE 
    WHEN COUNT(*) > 1000 THEN 'ALERT'
    WHEN COUNT(*) > 500 THEN 'WARNING'
    ELSE 'OK'
  END as status
FROM workspace.hive_video_analytics.silver_telemetry_quarantine
WHERE eventDate = CURRENT_DATE()
GROUP BY quarantine_reason
ORDER BY issue_count DESC;

## Recommended Alert Configurations

| Alert Name | Refresh Schedule | Condition | Severity | Recipients |
|------------|-----------------|-----------|----------|------------|
| **Data Freshness** | Every 15 minutes | hours_since_last_event > 2 | CRITICAL | On-call team, pipeline-alerts Slack channel |
| **High Quarantine Rate** | Every 30 minutes | quarantine_rate_pct > 5 | HIGH | Data engineering team, data-quality Slack channel |
| **Data Volume Anomaly** | Every 1 hour | pct_of_baseline < 50 | HIGH | Data engineering team |
| **Buffering Degradation** | Every 1 hour | avg_buffering_time_sec > 10 OR buffering_sessions_pct > 15 | MEDIUM | Video engineering team |
| **Gold Layer Lag** | Every 30 minutes | lag_hours > 1 | MEDIUM | Data engineering team |
| **Quarantine Reason Spike** | Every 1 hour | issue_count > 1000 | MEDIUM | Data quality team |

## Next Steps

1. ✅ Queries created in this notebook
2. ⬜ Create SQL alerts from each query (use "Create Alert" button)
3. ⬜ Configure notification channels (email, Slack, PagerDuty)
4. ⬜ Set up alert escalation policies
5. ⬜ Create runbook for each alert type
6. ⬜ Test alerts by triggering test conditions
7. ⬜ Document alert thresholds and tune based on production patterns

## How to Create SQL Alerts from These Queries

### Step 1: Save Query as a SQL Query Asset

1. Click on any SQL cell above (e.g., Alert 1: Data Freshness Check)
2. Click the **three dots menu (⋮)** in the top-right of the cell
3. Select **"Export" → "Save as SQL query"**
4. Give it a descriptive name (e.g., "Bronze Data Freshness Alert")
5. The query will open in a new SQL editor tab

### Step 2: Create Alert from the SQL Query

1. In the SQL query editor, run the query to verify it works
2. Click the **"Create Alert"** button (or use the dropdown next to Run)
3. Configure the alert:

#### Alert Settings:

**Refresh Schedule:**
* Choose how often the query runs (e.g., "Every 15 minutes")
* For critical alerts (data freshness), use shorter intervals
* For performance monitoring, hourly is often sufficient

**Alert Conditions:**
* Select the column to monitor (e.g., `hours_since_last_event` or `status`)
* Set the condition:
  - For numeric: "Value" > threshold (e.g., `hours_since_last_event > 2`)
  - For status: "Value" equals "ALERT"
* Set rearm seconds (time before alert can trigger again)

**Destinations:**
* Add notification recipients:
  - Email addresses
  - Slack channels (requires integration setup)
  - PagerDuty (requires integration setup)
  - Microsoft Teams (requires integration setup)

### Step 3: Test and Enable

1. **Test the alert** - Use "Send test notification" to verify recipients receive it
2. **Enable the alert** - Toggle the alert to "Enabled" status
3. **Monitor alert history** - Check the alert's history tab to see when it triggers

### Step 4: Document and Maintain

* Create a runbook for each alert explaining:
  - What triggers the alert
  - What actions to take
  - Who to escalate to
  - How to investigate the issue
* Review alert thresholds quarterly based on production patterns
* Tune false positive rates by adjusting thresholds

---

## Quick Start: Create Your First Alert

**Let's create the Data Freshness alert:**

1. Export [Cell 2: Alert 1: Data Freshness Check](#cell-1cb3083c-8ca0-44a0-9d93-3432a1dff197) as SQL query
2. Name it: `Video Telemetry - Bronze Data Freshness`
3. Refresh: Every 15 minutes
4. Condition: `status` equals `ALERT`
5. Destination: Your email + `#pipeline-alerts` Slack channel
6. Rearm: 3600 seconds (1 hour) to avoid spam
7. Enable and test!

**Once you've created alerts, update the checklist in the last cell.**

## How Batch Pipeline Updates Gold Layer

### 🏗️ Architecture: Materialized View (No Explicit MERGE)

Your gold layer uses a **MATERIALIZED VIEW** instead of a streaming table with MERGE operations.

```
Bronze (1,132 raw records)
  ├─→ Silver Enriched (2,518 clean records) ─→ Gold Metrics (40 aggregates)
  └─→ Silver Quarantine (0 bad records) ✗ NOT used in gold
```

### 🔄 How Updates Work

**When Pipeline Runs:**

1. **Bronze Layer**: Ingests new JSON files via Auto Loader
2. **Silver Layer Processing**:
   - ✅ **Clean data** → `silver_telemetry_enriched` (with quality checks)
   - ❌ **Bad data** → `silver_telemetry_quarantine` (isolated)
3. **Gold Layer Auto-Refresh**:
   - Materialized view **automatically detects** changes in `silver_telemetry_enriched`
   - **Incrementally recomputes** only affected date partitions
   - **No explicit MERGE** - Databricks handles updates internally

### ⚠️ Quarantine Data Behavior

**Key Point**: Quarantine data **NEVER flows to gold layer**

- Quarantine table is **separate** from the main data pipeline
- Gold reads **only** from `silver_telemetry_enriched`
- Quarantined records are excluded from metrics and aggregations

**Example with bad data:**
```
Pipeline run processes 1000 bronze records:
  → 950 pass quality checks → silver_telemetry_enriched → gold (included)
  → 50 fail quality checks → silver_telemetry_quarantine (excluded)
  
Gold metrics reflect only the 950 clean records
```

### 🎯 Incremental Refresh Logic

**Liquid Clustering**: `[eventDate, customerId, clientId]`

**Scenario 1: New data arrives**
```
2025-11-15 new data → Silver enriched adds 500 records for 2025-11-15
                   → Gold recomputes ONLY 2025-11-15 aggregates
                   → Other dates unchanged
```

**Scenario 2: Late data correction**
```
2025-11-10 late arrival → Silver enriched adds 50 records for 2025-11-10
                        → Gold recomputes ONLY 2025-11-10 aggregates
                        → Handles late data of any age (no watermark)
```

**Scenario 3: Mixed clean + quarantine**
```
1000 new records:
  → 980 clean → silver_enriched → gold includes these 980
  → 20 bad → silver_quarantine → gold excludes these 20
  
Gold aggregates reflect only 980 clean records per date
```

### 📊 Current Data Distribution

| Layer | Records | % of Bronze | Flow |
|-------|---------|-------------|------|
| Bronze (raw) | 1,132 | 100% | Input |
| Silver Enriched | 2,518 | 222% | Clean → Gold ✅ |
| Silver Quarantine | 0 | 0% | Bad → Isolated ❌ |
| Gold Aggregated | 40 | 3.5% | Final metrics |

**Why silver > bronze?** Quality distribution explosion (one bronze record → multiple silver records, one per video quality level)

**Why gold < silver?** Aggregation by `customer + client + date` reduces granularity

### 🔧 No MERGE Needed Because:

1. **Materialized views** handle incremental updates automatically
2. **Aggregations** (GROUP BY) naturally deduplicate and combine
3. **Date clustering** enables efficient partition-level recomputes
4. **Delta Lake** manages versioning and ACID transactions

### 🚀 Benefits vs Explicit MERGE

| Aspect | Materialized View | Explicit MERGE |
|--------|------------------|----------------|
| Code complexity | Simple aggregation | Complex MERGE logic |
| Late data handling | Automatic recompute | Manual watermark/state |
| Incremental refresh | Built-in | Manual implementation |
| Deduplication | Automatic (GROUP BY) | Manual MERGE keys |
| Maintenance | Low | High |

---

**Recommendation**: Keep the materialized view approach for this aggregation use case. Use explicit MERGE only if you need SCD Type 2 or CDC patterns.

In [0]:
%sql
-- Verify gold layer is up-to-date with silver
-- Shows if materialized view refresh is working correctly

WITH silver_dates AS (
  SELECT 
    MIN(eventDate) as earliest_date,
    MAX(eventDate) as latest_date,
    COUNT(DISTINCT eventDate) as date_count,
    COUNT(*) as record_count
  FROM workspace.hive_video_analytics.silver_telemetry_enriched
),
gold_dates AS (
  SELECT 
    MIN(eventDate) as earliest_date,
    MAX(eventDate) as latest_date,
    COUNT(DISTINCT eventDate) as date_count,
    COUNT(*) as record_count
  FROM workspace.hive_video_analytics.gold_viewer_qos_metrics
)
SELECT 
  'Silver Enriched' as layer,
  silver_dates.earliest_date,
  silver_dates.latest_date,
  silver_dates.date_count as distinct_dates,
  silver_dates.record_count
FROM silver_dates

UNION ALL

SELECT 
  'Gold Metrics',
  gold_dates.earliest_date,
  gold_dates.latest_date,
  gold_dates.date_count,
  gold_dates.record_count
FROM gold_dates

UNION ALL

SELECT
  '⚠️ Date Gap',
  NULL,
  NULL,
  NULL,
  CASE 
    WHEN gold_dates.latest_date < silver_dates.latest_date 
    THEN DATEDIFF(silver_dates.latest_date, gold_dates.latest_date)
    ELSE 0
  END as days_behind
FROM silver_dates, gold_dates;